# Logistic Regression for Classification

This notebook trains a **Logistic Regression classifier** on the Wisconsin Breast Cancer Diagnostic dataset, evaluates it with standard classification metrics, and plots a probability (sigmoid) curve, ROC curve, and confusion-matrix heatmap.

**Dataset:** `sklearn.datasets.load_breast_cancer` — 569 samples and 30 numeric features; the dataset is a copy of the UCI Breast Cancer Wisconsin (Diagnostic) dataset. Source: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

**Positive class:** malignant (1). **Negative class:** benign (0). This is a teaching example, not a medical diagnostic tool.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42

## 1. Load and prepare the dataset

The original target uses `0 = malignant` and `1 = benign`. We remap it so malignant is the positive class (`1`), making malignant-case recall easy to interpret. The split is stratified to preserve the class proportions. Standardization is fitted only on the training data through a pipeline to prevent data leakage.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
y = (dataset.target == 0).astype(int)  # 1 = malignant, 0 = benign

print('Dataset shape:', X.shape)
print('Class counts (0 = benign, 1 = malignant):')
print(y.value_counts().sort_index())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('\nTraining samples:', len(X_train))
print('Testing samples:', len(X_test))

## 2. Train the Logistic Regression model

`StandardScaler` standardizes the feature values. `LogisticRegression` learns the class boundary and estimates class probabilities.

In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]  # probability of malignant class
print('Training complete.')

## 3. Evaluation metrics

Accuracy is overall correctness; precision measures how often malignant predictions are correct; recall measures the fraction of malignant cases detected; F1 balances precision and recall; ROC-AUC measures ranking quality across thresholds.

In [1]:
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f} ({accuracy_score(y_test, y_pred):.2%})')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'F1-score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_probability):.4f}')
print('\nClassification report (0 = benign, 1 = malignant):')
print(classification_report(y_test, y_pred, target_names=['benign', 'malignant']))

Accuracy : 0.9649 (96.49%)
Precision: 0.9750
Recall   : 0.9286
F1-score : 0.9512
ROC-AUC  : 0.9960

Classification report (0 = benign, 1 = malignant):
              precision    recall  f1-score   support

      benign       0.96      0.99      0.97        72
   malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


## 4. Logistic probability (sigmoid) curve

The model uses all 30 features, so its probability depends on all features together. To visualize a one-dimensional curve, this plot varies **mean radius** while holding every other feature at its training-set median. The result is a conditional view of how the fitted model's malignant probability changes with mean radius; it is not a separate one-feature model.

In [ ]:
feature_name = 'mean radius'
feature_values = np.linspace(X[feature_name].min(), X[feature_name].max(), 300)
reference_row = X_train.median().to_frame().T
curve_data = pd.concat([reference_row] * len(feature_values), ignore_index=True)
curve_data[feature_name] = feature_values
malignant_probability = model.predict_proba(curve_data)[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(feature_values, malignant_probability, linewidth=2, label='Predicted P(malignant)')
plt.axhline(0.5, linestyle='--', label='Decision threshold = 0.5')
plt.xlabel('Mean radius')
plt.ylabel('Predicted probability of malignant class')
plt.title('Logistic Regression Probability Curve')
plt.ylim(-0.03, 1.03)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 5. ROC curve

The ROC curve plots the true-positive rate against the false-positive rate at different probability thresholds. The AUC is calculated from the held-out test set.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_probability)
auc_value = roc_auc_score(y_test, y_probability)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, linewidth=2, label=f'Logistic Regression (AUC = {auc_value:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Test Set')
plt.grid(alpha=0.3)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Confusion matrix

Rows are true labels and columns are predicted labels. Label order is benign, then malignant.

In [2]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print('Confusion matrix (rows = true; columns = predicted):')
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign', 'Malignant'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()

Confusion matrix (rows = true; columns = predicted):
[[71  1]
 [ 3 39]]


## Conclusion

With the fixed stratified 80/20 split (`random_state=42`), the notebook's saved reference output shows **96.49% accuracy** and **0.996 ROC-AUC**. Re-run all cells in Jupyter or Google Colab to regenerate the plots and verify results in your installed library versions. Because this is a medical dataset, these metrics are for learning only and do not establish clinical usefulness.